In [1]:
import numpy as np

In [2]:
import math, random
import cvxpy as cp

In [3]:
def solve_joint_measurement_2x2():
    Ma0 = cp.Variable((2, 2), hermitian=True)
    Ma1 = cp.Variable((2, 2), hermitian=True)
    Mb0 = cp.Variable((2, 2), hermitian=True)
    Mb1 = cp.Variable((2, 2), hermitian=True)

    # Define Pauli matrices as constants
    sigma_X = cp.Constant(np.array([[0, 1], [1, 0]], dtype=complex))
    sigma_Z = cp.Constant(np.array([[1, 0], [0, -1]], dtype=complex))

    eta = cp.Variable(nonneg=True)

    N = [[cp.Variable((2, 2), hermitian=True) for _ in range(2)] for _ in range(2)]

    constraints = [
        Ma0 == 0.5 * (np.eye(2) + eta * sigma_X),
        Ma1 == 0.5 * (np.eye(2) - eta * sigma_X),
        Mb0 == 0.5 * (np.eye(2) + eta * sigma_Z),
        Mb1 == 0.5 * (np.eye(2) - eta * sigma_Z),
        N[0][0] + N[0][1] == Ma0,
        N[1][0] + N[1][1] == Ma1,
        N[0][0] + N[1][0] == Mb0,
        N[0][1] + N[1][1] == Mb1,
        eta <= 1,
        N[0][0] >> 0,
        N[0][1] >> 0,
        N[1][0] >> 0,
        N[1][1] >> 0,
        N[0][0] + N[0][1] + N[1][0] + N[1][1] == np.eye(2)
    ]

    prob = cp.Problem(cp.Maximize(eta), constraints)
    prob.solve()

sigma_X = cp.Constant(np.array([[0, 1], [1, 0]], dtype=complex))
sigma_Z = cp.Constant(np.array([[1, 0], [0, -1]], dtype=complex))

In [ ]:
solve_joint_measurement_2x2()


In [5]:
# import cvxpy as cp
print(cp.__version__, hasattr(cp, "promote"))

1.8.2 True


In [4]:
# stepping stone for generalization to higher dimensions

def solve_joint_measurement_general_old(d = 2, cst: list[cp.Constant] = [sigma_X, sigma_Z]):
    Ma0 = cp.Variable((d, d), hermitian=True)
    Ma1 = cp.Variable((d, d), hermitian=True)
    Mb0 = cp.Variable((d, d), hermitian=True)
    Mb1 = cp.Variable((d, d), hermitian=True)

    # Define Pauli matrices as constants
    A = cst[0]
    B = cst[1]

    eta = cp.Variable(nonneg=True)

    N = [[cp.Variable((d, d), hermitian=True) for _ in range(2)] for _ in range(2)]

    constraints = [
        Ma0 == 0.5 * (np.eye(d) + eta * A),
        Ma1 == 0.5 * (np.eye(d) - eta * A),
        Mb0 == 0.5 * (np.eye(d) + eta * B),
        Mb1 == 0.5 * (np.eye(d) - eta * B),
        N[0][0] + N[0][1] == Ma0,
        N[1][0] + N[1][1] == Ma1,
        N[0][0] + N[1][0] == Mb0,
        N[0][1] + N[1][1] == Mb1,
        eta <= 1,
        N[0][0] >> 0,
        N[0][1] >> 0,
        N[1][0] >> 0,
        N[1][1] >> 0,
        N[0][0] + N[0][1] + N[1][0] + N[1][1] == np.eye(d)
    ]

    prob = cp.Problem(cp.Maximize(eta), constraints)
    print(prob.solve())



In [5]:
A_op = np.diag([1, 0, -1])
U = np.array([
    [1, 1, 1],
    [1, np.exp(2j*np.pi/3), np.exp(4j*np.pi/3)],
    [1, np.exp(4j*np.pi/3), np.exp(2j*np.pi/3)]
], dtype=complex) / np.sqrt(3)

B_op = U @ A_op @ U.conj().T

solve_joint_measurement_general_old(d = 3, cst=[A_op, B_op])

0.7962074511760084


In [6]:
# more testing 
lam1 = np.array([
    [0, 1, 0],
    [1, 0, 0],
    [0, 0, 0]
], dtype=complex)

lam3 = np.array([
    [1, 0, 0],
    [0, -1, 0],
    [0, 0, 0]
], dtype=complex)

A_op = cp.Constant(lam1)
B_op = cp.Constant(lam3)

I3 = np.eye(3)

solve_joint_measurement_general_old(d = 3, cst=[A_op, B_op])

0.7071155421843797


In [7]:
lam8 = (1/np.sqrt(3)) * np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, -2]
], dtype=complex)

B_op = cp.Constant(lam8)

solve_joint_measurement_general_old(d = 3, cst=[A_op, B_op])

0.8660254037911704


In [8]:
import itertools

def build_lambdas (m, outcome_list):
    return list(itertools.product(outcome_list, repeat=m))

In [9]:
# test ts
build_lambdas(2, [-1, 1])

[(-1, -1), (-1, 1), (1, -1), (1, 1)]

In [10]:
def make_joint_povm(d, lambdas):
    return {
        lam: cp.Variable((d, d), hermitian=True)
        for lam in lambdas
    }


In [11]:
# has bugs

# def joint_constraints(M, N, lambdas, m, outcomes, d):
#     constraints = []

#     # positivity
#     for Nlam in N.values():
#         constraints.append(Nlam >> 0)

#     # normalization
#     constraints.append(cp.sum([Nlam for Nlam in N.values()]) == np.eye(d))

#     # marginals
#     for x in range(m):
#         for a in outcomes:
#             constraints.append(
#                 cp.sum([N[lam] for lam in lambdas if lam[x] == a])
#                 == M[a][x]
#             )
            

#     return constraints

In [12]:
def joint_constraints(M, N, lambdas, m, outcomes, d):
    constraints = []

    # positivity
    for Nlam in N.values():
        constraints.append(Nlam >> 0)

    # normalization
    total = cp.sum([Nlam for Nlam in N.values()])
    constraints.append(total == np.eye(d))

    # marginals — THIS is where the bug likely hides
    for x in range(m):
        for a in outcomes:
            matching = [N[lam] for lam in lambdas if lam[x] == a]
            marginal = cp.sum(matching)  # must be cp.sum, not sum()
            constraints.append(marginal == M[a][x])  # M[a][x] is affine in eta

    return constraints

In [13]:
def get_projectors(op_matrix, outcomes):
    # extract numpy array from cp.Constant if needed
    if isinstance(op_matrix, cp.Constant):
        op_matrix = op_matrix.value
    op_matrix = np.array(op_matrix)
    
    eigenvalues, eigenvectors = np.linalg.eigh(op_matrix)
    
    sort_idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[sort_idx]
    eigenvectors = eigenvectors[:, sort_idx]
    
    sorted_outcomes = sorted(outcomes)
    assert len(eigenvalues) == len(sorted_outcomes), \
        f"Eigenvalue count {len(eigenvalues)} != outcomes count {len(sorted_outcomes)}"
    
    projectors = {}
    for a, i in zip(sorted_outcomes, range(len(sorted_outcomes))):
        v = eigenvectors[:, i:i+1]
        projectors[a] = v @ v.conj().T
    
    P_sum = sum(projectors.values())
    assert np.allclose(P_sum, np.eye(op_matrix.shape[0]), atol=1e-8), \
        "Projectors don't sum to identity!"
    
    return projectors


def get_outcomes(cst):
    op = np.array(cst[0].value) if isinstance(cst[0], cp.Constant) else np.array(cst[0])
    eigenvalues = np.linalg.eigvalsh(op)
    # round to nearest integer only if eigenvalues are close to integers
    rounded = [round(float(e.real), 8) for e in eigenvalues]
    return sorted(set(rounded))  # set() removes duplicates from degenerate eigenvalues

In [14]:


def solve_joint_measurement_general(d=2, cst=None, m=2, outcomes=None):
    
    if outcomes is None:
        outcomes = get_outcomes(cst)
        print(f"Inferred outcomes: {outcomes}")
    
    eta = cp.Variable(nonneg=True)
    k = len(outcomes)

    # Build projectors for each observable
    all_projectors = []
    for x in range(m):
        op = np.array(cst[x].value) if isinstance(cst[x], cp.Constant) else np.array(cst[x])
        all_projectors.append(get_projectors(op, outcomes))

    # M[a][x] = (1-eta)/k * I + eta * P[a][x]
    M = {
        a: [
            (1/k) * np.eye(d) + eta * (all_projectors[x][a] - (1/k) * np.eye(d))
            for x in range(m)
        ]
        for a in outcomes
    }

    # Quick sanity check before solving
    for a in outcomes:
        for x in range(m):
            expr = M[a][x]
            print(f"M[{a}][{x}] shape: {expr.shape}, is affine: {expr.is_affine()}")
            
    # Also check projectors sum to identity
    for x in range(m):
        P_sum = sum(all_projectors[x][a] for a in outcomes)
        print(f"Projectors x={x} sum to I: {np.allclose(P_sum, np.eye(d))}")



    lambdas = build_lambdas(m, outcomes)
    N = make_joint_povm(d, lambdas)

    constraints = [eta <= 1]
    constraints += joint_constraints(M, N, lambdas, m, outcomes, d)

    prob = cp.Problem(cp.Maximize(eta), constraints)
    prob.solve(solver=cp.SCS)
    print(f"Solver status: {prob.status}")
    print(f"eta* = {eta.value:.6f}")
    return eta.value

In [15]:
solve_joint_measurement_general(d=3, cst=[A_op, B_op], m=2, outcomes=[-1, 0, 1])

M[-1][0] shape: (3, 3), is affine: True
M[-1][1] shape: (3, 3), is affine: True
M[0][0] shape: (3, 3), is affine: True
M[0][1] shape: (3, 3), is affine: True
M[1][0] shape: (3, 3), is affine: True
M[1][1] shape: (3, 3), is affine: True
Projectors x=0 sum to I: True
Projectors x=1 sum to I: True
Solver status: optimal
eta* = 0.660193


np.float64(0.6601925727538678)

In [16]:
solve_joint_measurement_general(d=2, cst=[sigma_X, sigma_Z], m=2, outcomes=[-1, 1])

M[-1][0] shape: (2, 2), is affine: True
M[-1][1] shape: (2, 2), is affine: True
M[1][0] shape: (2, 2), is affine: True
M[1][1] shape: (2, 2), is affine: True
Projectors x=0 sum to I: True
Projectors x=1 sum to I: True
Solver status: optimal
eta* = 0.707107


np.float64(0.7071071157293095)

In [17]:
sigma_Y = cp.Constant(np.array([[0, -1j], [1j, 0]], dtype=complex))

solve_joint_measurement_general(d=2, cst=[sigma_X, sigma_Y, sigma_Z], m=3, outcomes=[-1, 1])


M[-1][0] shape: (2, 2), is affine: True
M[-1][1] shape: (2, 2), is affine: True
M[-1][2] shape: (2, 2), is affine: True
M[1][0] shape: (2, 2), is affine: True
M[1][1] shape: (2, 2), is affine: True
M[1][2] shape: (2, 2), is affine: True
Projectors x=0 sum to I: True
Projectors x=1 sum to I: True
Projectors x=2 sum to I: True
Solver status: optimal
eta* = 0.577337


np.float64(0.5773369757847167)

In [18]:
solve_joint_measurement_general(d = 3, cst=[A_op, B_op])

Inferred outcomes: [-1.0, 0.0, 1.0]
M[-1.0][0] shape: (3, 3), is affine: True
M[-1.0][1] shape: (3, 3), is affine: True
M[0.0][0] shape: (3, 3), is affine: True
M[0.0][1] shape: (3, 3), is affine: True
M[1.0][0] shape: (3, 3), is affine: True
M[1.0][1] shape: (3, 3), is affine: True
Projectors x=0 sum to I: True
Projectors x=1 sum to I: True
Solver status: optimal
eta* = 0.660193


np.float64(0.6601925727538678)

In [ ]:
1/math.sqrt(3)

0.5773502691896258

: 